# R₀ Calculation for SEIRS-SEI Model

This notebook derives the basic reproduction number $R_0$ for a vector-borne disease model with:

- **Human side (SEIRS):** Susceptible → Exposed → Infected → Recovered → Susceptible
- **Mosquito side (SEI):** Susceptible → Exposed → Infected
- **Delays:** $\tau_H$ (human incubation) and $\tau_M$ (sporogonic cycle)

## Explicit DDEs (as in Test_SEIRS_SEI_Model.ipynb)

Based on the actual implementation:

### Human (SEIRS):
$$\frac{dS_H}{dt} = r_H N - a b_2 \frac{I_M(t)}{N} S_H(t) + \omega R_H$$
$$\frac{dE_H}{dt} = a b_2 \frac{I_M(t)}{N} S_H(t) - a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H)$$
$$\frac{dI_H}{dt} = a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H) - \gamma I_H$$
$$\frac{dR_H}{dt} = \gamma I_H - \omega R_H$$

### Mosquito (SEI):
$$\frac{dS_M}{dt} = \Lambda - a b_1 \frac{I_H(t)}{N} S_M(t) - \mu S_M(t)$$
$$\frac{dE_M}{dt} = a b_1 \frac{I_H(t)}{N} S_M(t) - a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell - \mu E_M(t)$$
$$\frac{dI_M}{dt} = a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell - \mu I_M(t)$$

where $\ell = e^{-\mu \tau_M}$ is the probability of surviving the sporogonic delay.

**Note:** Force of infection uses **current** values for new exposures, progression to infectious uses **delayed** values.

**Mosquito recruitment:**
$$\Lambda = b(R,T) \cdot K \cdot \left(1 - \frac{S_M + E_M + I_M}{K}\right)$$

**Parameters:**
- $r_H$: Human net growth rate
- $a$: Mosquito biting rate
- $b_1$: Probability of transmitting from human to mosquito
- $b_2$: Probability of transmitting from mosquito to human
- $\tau_H$: Human incubation period
- $\tau_M$: Sporogonic delay
- $\ell = e^{-\mu \tau_M}$: Probability of surviving sporogony
- $b(R,T)$: Temperature/rain dependent recruitment
- $K$: Mosquito carrying capacity: $K = M \times $ Temperature factor $ \times $ Rain factor $ \times $ Seasonal factor
  - Temperature factor: $\max\left(0.1, \min\left(1, \dfrac{T - 15}{15}\right)\right)$
  - Rain factor: $\min\left(1, \max\left(0, \dfrac{R}{30}\right)\right)$
  - Seasonal factor: $2.5 + 2\sin\left(2\pi\dfrac{t - 180}{365.25}\right) \ (1\leq t \leq 366)$
- $\mu$: Mosquito mortality rate
- $\omega$: Immunity loss rate
- $\gamma$: Recovery rate

## Methodology: Next Generation Matrix Approach

**Infected compartments:** $[E_H, I_H, E_M, I_M]$

Define:
- $\mathcal{F}$: rate of new infections
- $\mathcal{V} = \mathcal{V}^{-} - \mathcal{V}^{+}$: rate of transitions out minus in

At the **Disease-Free Equilibrium (DFE)**:
- $S_H^* = N, E_H^* = I_H^* = R_H^* = 0$
- $S_M^* = M, E_M^* = I_M^* = 0$

Then $R_0 = \rho(FV^{-1})$ where $\rho$ is the spectral radius.

### $\mathcal{F}$ (New Infections)

Only terms that create **newly infected** individuals:

- $\mathcal{F}_1 = a b_2 \frac{I_M}{N} S_H$ → at DFE: $a b_2 I_M$
- $\mathcal{F}_2 = 0$
- $\mathcal{F}_3 = a b_1 \frac{I_H}{N} S_M$ → at DFE: $a b_1 \frac{M}{N} I_H$
- $\mathcal{F}_4 = 0$

### $\mathcal{V} = \mathcal{V}^{-} - \mathcal{V}^{+}$ (Transitions out minus in)

**For $E_H$:**
- $\mathcal{V}_1^{-} = a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H)$ (outflow to $I_H$)
- $\mathcal{V}_1^{+} = 0$
- $\mathcal{V}_1 = \mathcal{V}_1^{-} - \mathcal{V}_1^{+} = a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H)$

**For $I_H$:**
- $\mathcal{V}_2^{-} = \gamma I_H$ (recovery)
- $\mathcal{V}_2^{+} = a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H)$ (inflow from $E_H$)
- $\mathcal{V}_2 = \gamma I_H - a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H)$

**For $E_M$:**
- $\mathcal{V}_3^{-} = a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell + \mu E_M$ (to $I_M$ + death)
- $\mathcal{V}_3^{+} = a b_1 \frac{I_H(t)}{N} S_M(t)$ (inflow from $S_M$)
- $\mathcal{V}_3 = a b_1 \frac{I_H(t)}{N} S_M - a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell - \mu E_M$

**For $I_M$:**
- $\mathcal{V}_4^{-} = \mu I_M$ (death)
- $\mathcal{V}_4^{+} = a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell$ (inflow from $E_M$)
- $\mathcal{V}_4 = \mu I_M - a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell$

## R₀ for SEIRS

At DFE ($S_H^* = N$):
- New infections enter via $E_H$: $\mathcal{F}_1 = a b_2 I_M$
- $\mathcal{V}_1 = \sigma_H E_H$ (outflow to $I_H$)
- $\mathcal{V}_2 = \gamma I_H - \sigma_H E_H$

$$F = \begin{pmatrix} 0 & a b_2 \end{pmatrix}, \quad V = \begin{pmatrix} \sigma_H & 0 \\ -\sigma_H & \gamma \end{pmatrix}$$

$$R_0^{human} = \frac{a b_2}{\gamma}$$

Note: $\tau_H$ cancels out because exposed compartment only delays progression.

## R₀ for SEI

At DFE ($S_M^* = M$):
- New infections enter via $E_M$: $\mathcal{F}_1 = a b_1 \frac{M}{N} I_H$
- Outflow to $I_M$ includes survival factor $\ell = e^{-\mu \tau_M}$

$$F = \begin{pmatrix} 0 & \frac{a b_1 M \ell}{N} \end{pmatrix}, \quad V = \begin{pmatrix} \mu & 0 \\ 0 & \mu \end{pmatrix}$$

$$R_0^{mosquito} = \frac{a b_1 M \ell}{N \mu} = \frac{a b_1 M}{N} \cdot \frac{e^{-\mu \tau_M}}{\mu}$$

Here $\tau_M$ explicitly appears via $\ell = e^{-\mu \tau_M}$.

## Final R₀ (SEIRS-SEI Combined)

### F Jacobian (∂ℱᵢ/∂xⱼ at DFE)

State vector order: $[E_H, I_H, E_M, I_M]$

$$\mathcal{F} = \begin{pmatrix} a b_2 I_M \\ 0 \\ a b_1 \frac{M}{N} I_H \\ 0 \end{pmatrix}$$

$$F = \begin{pmatrix} 
0 & 0 & 0 & a b_2 \\
0 & 0 & 0 & 0 \\
0 & a b_1 \frac{M}{N} & 0 & 0 \\
0 & 0 & 0 & 0
\end{pmatrix}$$

### V Jacobian (∂𝒱ᵢ/∂xⱼ at DFE)

Evaluating partial derivatives of $\mathcal{V}$ at DFE ($E_H=I_H=E_M=I_M=0$):

$$\mathcal{V} = \begin{pmatrix} 
a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H) \\
\gamma I_H - a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H) \\
a b_1 \frac{I_H(t)}{N} S_M - a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell - \mu E_M \\
\mu I_M - a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell
\end{pmatrix}$$

At DFE ($S_H=N, S_M=M$):

$$V = \begin{pmatrix} 
0 & 0 & 0 & 0 \\
0 & \gamma & 0 & 0 \\
0 & a b_1 \frac{M}{N} & \mu & 0 \\
0 & 0 & 0 & \mu
\end{pmatrix}$$

**Note:** The delayed terms in $\mathcal{V}$ don't contribute to the Jacobian at DFE because they involve products of state variables that are zero at DFE.

$K = F V^{-1}$

First, compute $V^{-1}$:

$$V^{-1} = \begin{pmatrix} 
0 & 0 & 0 & 0 \\
0 & 1/\gamma & 0 & 0 \\
0 & -\frac{a b_1 M}{\mu N \gamma} & 1/\mu & 0 \\
0 & 0 & 0 & 1/\mu
\end{pmatrix}$$

Then:

$$K = F V^{-1} = \begin{pmatrix} 
0 & 0 & \frac{a b_2}{\mu} & \frac{a b_2}{\mu} \\
0 & 0 & 0 & 0 \\
0 & \frac{a b_1 M}{N \gamma} & 0 & 0 \\
0 & 0 & 0 & 0
\end{pmatrix}$$

The eigenvalues are: $0, 0, \pm \sqrt{\frac{a^2 b_1 b_2 M \ell}{N \gamma \mu}}$

The spectral radius is:

$$R_0 = \sqrt{ \frac{a^2 b_1 b_2 M \ell}{N \gamma \mu} } = \frac{a \sqrt{b_1 b_2 M/N}}{\sqrt{\gamma \mu}} \cdot \sqrt{\ell}$$

where $\ell = e^{-\mu \tau_M}$ is the probability of surviving the sporogonic delay.

This can be written as:

$$R_0 = \sqrt{R_0^{H} \cdot R_0^{M}}$$

where:
- $R_0^{H} = \frac{a b_2}{\gamma}$ 
- $R_0^{M} = \frac{a b_1 M}{N} \cdot \frac{\ell}{\mu}$ 

## Summary Table

| Model | Formula |
|-------|---------|
**SEIRS** | $R_0^{H} = \frac{a b_2}{\gamma}$ |
**SEI** | $R_0^{M} = \frac{a b_1 M}{N} \cdot \frac{e^{-\mu \tau_M}}{\mu}$ |
**SEIRS-SEI** | $R_0 = \sqrt{R_0^{H} \cdot R_0^{M}}$ |

**Key insight:** $\tau_H$ cancels out (no mortality during incubation). $\tau_M$ reduces $R_0$ via survival factor $e^{-\mu \tau_M}$.